# StayNest - Session 7 Assignment (Delta Lake & Lakehouse)
Work through the 8 tasks in order. Read the Assignment Questions PDF for the full
detail and acceptance criteria. Fill each `# TODO` cell, run it, and keep the output
visible. Runs on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `bookings_updates.csv` to a Volume, set `BASE`,
`CATALOG`, `SCHEMA`, and run this cell. Expect 12000 / 200 / 200.

In [0]:
BASE    = "/Volumes/workspace/default/staynest"
CATALOG = "workspace"
SCHEMA  = "default"
FQN = lambda name: f"{CATALOG}.{SCHEMA}.{name}"

read_csv = lambda name: (spark.read
    .option("header", True).option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df = read_csv("bookings")
hotels_df   = read_csv("hotels")
updates_df  = read_csv("bookings_updates")

print(f"bookings: {bookings_df.count()}, hotels: {hotels_df.count()}, "
      f"updates: {updates_df.count()}")

bookings: 12000, hotels: 200, updates: 200


## Task 1 - Read the plan and force a broadcast join
Join bookings to hotels and call `.explain()` to see the plan. Then force a
broadcast join with `broadcast(hotels_df)` and `.explain()` again. In a comment,
say which join each plan used and why broadcast avoids a shuffle.
(Tip: hotels also has a `city` column, so `hotels_df.drop("city")` before joining.)

In [0]:
from pyspark.sql.functions import broadcast, col, count, sum as _sum, current_timestamp
h = hotels_df.drop("city")

print("=== PLAIN JOIN ===")
bookings_df.join(h, "hotel_id").explain()
print("=== BROADCAST JOIN ===")
bookings_df.join(broadcast(h), "hotel_id").explain()

=== PLAIN JOIN ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [hotel_id#10078, booking_id#10076, customer_id#10077, booking_date#10079, city#10080, nights#10081, amount#10082, status#10083, hotel_name#10108, category#10110, star_rating#10111]
         +- PhotonBroadcastHashJoin [hotel_id#10078], [hotel_id#10107], Inner, BuildRight, false, true, false
            :- PhotonFilter isnotnull(hotel_id#10078)
            :  +- PhotonRowToColumnar
            :     +- FileScan csv [booking_id#10076,customer_id#10077,hotel_id#10078,booking_date#10079,city#10080,nights#10081,amount#10082,status#10083] Batched: false, DataFilters: [isnotnull(hotel_id#10078)], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/staynest/bookings.csv], PartitionFilters: [], PushedFilters: [IsNotNull(hotel_id)], ReadSchema: struct<booking_id:int,customer_id:int,hotel_id:int,booking_

## Task 2 - Create a Delta table, then read its history
Write `bookings_df` as a managed Delta table with `saveAsTable`. Then create some
history: run an `UPDATE` (set pending to completed) and a `DELETE` (remove
cancelled). Show `DESCRIBE HISTORY` and point out the versioned commits.

In [0]:
T = FQN("bookings_delta")
bookings_df.write.mode("overwrite").format("delta").saveAsTable(T)
spark.sql(f"UPDATE {T} SET status = 'completed' WHERE status = 'pending'")
spark.sql(f"DELETE FROM {T} WHERE status = 'cancelled'")
display(spark.sql(f"DESCRIBE HISTORY {T}").select("version","timestamp","operation"))
# v0 = WRITE, v1 = UPDATE, v2 = DELETE: separate versioned commits

version,timestamp,operation
2,2026-10-12T06:31:38.000Z,DELETE
1,2026-10-12T06:31:34.000Z,UPDATE
0,2026-10-12T06:31:26.000Z,CREATE OR REPLACE TABLE AS SELECT


## Task 3 - Time travel and RESTORE
Read the table as it was at **version 0** (before your UPDATE and DELETE) and show
its count. Then `RESTORE` the table to version 0 and confirm the count is back.
Show that RESTORE appears as a new commit in the history.

In [0]:
print("version 0 count:", spark.read.option("versionAsOf", 0).table(T).count())
print("current count  :", spark.table(T).count())
spark.sql(f"RESTORE TABLE {T} TO VERSION AS OF 0")
print("after RESTORE  :", spark.table(T).count())
display(spark.sql(f"DESCRIBE HISTORY {T}").select("version","timestamp","operation"))
# RESTORE is a new commit (v3), so it is non-destructive: old versions stay in the log.

version 0 count: 12000
current count  : 10563
after RESTORE  : 12000


version,timestamp,operation
4,2026-10-12T06:31:56.000Z,RESTORE
3,2026-10-12T06:31:42.000Z,OPTIMIZE
2,2026-10-12T06:31:38.000Z,DELETE
1,2026-10-12T06:31:34.000Z,UPDATE
0,2026-10-12T06:31:26.000Z,CREATE OR REPLACE TABLE AS SELECT


## Task 4 - OPTIMIZE and ZORDER
Run `OPTIMIZE` on your Delta table to compact files. Then run
`OPTIMIZE ... ZORDER BY (city)`. In a comment, say what OPTIMIZE does and why
`city` is a good ZORDER column but `status` would not be.

In [0]:
display(spark.sql(f"OPTIMIZE {T}"))
display(spark.sql(f"OPTIMIZE {T} ZORDER BY (city)"))
# OPTIMIZE compacts many small files into fewer large ones, so reads scan less.
# city: high cardinality and often filtered, so ZORDER enables data skipping.
# status: only 3 values, so every file would contain all of them and nothing can be skipped.

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1791786724131, 1791786726329, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null, 0)"


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 113162), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1791786728000, 1791786730693, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null, 0)"


## Task 5 - Bronze: land the raw data
Write the raw bookings to a `bronze_bookings` Delta table, keeping every row and
adding an `ingested_at` timestamp column.

In [0]:
(bookings_df.withColumn("ingested_at", current_timestamp())
    .write.mode("overwrite").format("delta").saveAsTable(FQN("bronze_bookings")))
print(spark.table(FQN("bronze_bookings")).count())   # 12000
display(spark.table(FQN("bronze_bookings")).limit(5))

12000


booking_id,customer_id,hotel_id,booking_date,city,nights,amount,status,ingested_at
9000000,701600,3095,2025-11-27,Jaipur,4,6087.65,completed,2026-10-12T06:32:38.197Z
9000001,700065,3057,2025-11-06,Delhi,1,8211.19,cancelled,2026-10-12T06:32:38.197Z
9000002,701392,3187,2025-08-21,Jaipur,2,7176.52,cancelled,2026-10-12T06:32:38.197Z
9000003,700867,3112,2025-03-22,Bengaluru,5,7880.62,completed,2026-10-12T06:32:38.197Z
9000004,701521,3043,2025-04-19,Mumbai,5,21021.51,pending,2026-10-12T06:32:38.197Z


## Task 6 - Silver: clean and conform
Build `silver_bookings` from bronze: keep only completed bookings and join the
hotel dimension to add `category`, `star_rating`, and the hotel name. Drop the
duplicate `city` from the hotel side so the join has a single `city`.

In [0]:
silver = (spark.table(FQN("bronze_bookings"))
    .filter(col("status") == "completed")
    .join(hotels_df.drop("city"), "hotel_id"))
silver.write.mode("overwrite").format("delta").saveAsTable(FQN("silver_bookings"))
print(spark.table(FQN("silver_bookings")).columns)   # single city column
display(spark.table(FQN("silver_bookings")).limit(5))

['hotel_id', 'booking_id', 'customer_id', 'booking_date', 'city', 'nights', 'amount', 'status', 'ingested_at', 'hotel_name', 'category', 'star_rating']


hotel_id,booking_id,customer_id,booking_date,city,nights,amount,status,ingested_at,hotel_name,category,star_rating
3095,9000000,701600,2025-11-27,Jaipur,4,6087.65,completed,2026-10-12T06:32:38.197Z,Orchid Suites,Budget,3.8
3112,9000003,700867,2025-03-22,Bengaluru,5,7880.62,completed,2026-10-12T06:32:38.197Z,Grand Stay,Budget,3.6
3012,9000006,701336,2025-11-25,Delhi,7,70999.15,completed,2026-10-12T06:32:38.197Z,Orchid Residency,Luxury,4.1
3127,9000007,700868,2025-04-10,Mumbai,2,15693.64,completed,2026-10-12T06:32:38.197Z,Summit Stay,Luxury,4.5
3045,9000008,700687,2025-01-15,Mumbai,1,1492.62,completed,2026-10-12T06:32:38.197Z,Sunset Inn,Budget,3.0


## Task 7 - Gold: business-ready aggregate
From silver, build a `gold_city_revenue` Delta table: bookings and total revenue
per city, ordered by revenue.

In [0]:
gold = (spark.table(FQN("silver_bookings")).groupBy("city")
    .agg(count("*").alias("bookings"), _sum("amount").alias("revenue"))
    .orderBy(col("revenue").desc()))
gold.write.mode("overwrite").format("delta").saveAsTable(FQN("gold_city_revenue"))
display(spark.table(FQN("gold_city_revenue")).orderBy(col("revenue").desc()))

city,bookings,revenue
Goa,2546,4.459670179E7
Mumbai,1715,3.624122111999999E7
Delhi,1174,2.631428154000002E7
Jaipur,979,2.4436853129999984E7
Bengaluru,1318,2.2670136969999984E7
Udaipur,691,1.2094427419999994E7
Rishikesh,407,8606121.57999999
Manali,480,6235480.679999996
Munnar,244,3979216.11
Anantapur,106,2257080.65


## Task 8 - Incremental load with MERGE
You have today's batch in `updates_df` (150 changed bookings + 50 new ones).
`MERGE` it into your Delta table: update matched booking_ids, insert new ones, in
one command. Report the row count before and after (it should grow by the 50 new).

In [0]:
before = spark.table(T).count()
updates_df.createOrReplaceTempView("batch")
spark.sql(f"""
    MERGE INTO {T} AS t
    USING batch AS s
    ON t.booking_id = s.booking_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
after = spark.table(T).count()
print(f"before: {before}, after: {after}, new rows: {after - before}")   # 12000 -> 12050

before: 12000, after: 12050, new rows: 50
